In [ ]:
# 주택 가격예측

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

# 1. 데이터 로드 및 전처리
file_path = '국토교통부_주택 공시가격 정보(2024).csv'

# 필요한 열 선택
features = ['시도', '시군구', '전용면적']
target = '공시가격'

# 결측치 제거
df.dropna(subset=features + [target], inplace=True)

# 2. 범주형 변수 전처리 (원-핫 인코딩)
categorical_features = ['시도', '시군구']
numerical_features = ['전용면적']

# OneHotEncoder의 sparse_output=False를 사용하여 항상 밀집 배열 반환
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
        ('num', 'passthrough', numerical_features)
    ])

# 3. 데이터 분할
X = df[features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 파이프라인을 사용하여 데이터 변환
# toarray() 호출이 필요 없습니다.
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# 4. 텐서플로우 모델 구축
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_transformed.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1, activation='linear')
])

# 모델 컴파일
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

# 모델 요약
model.summary()

# 5. 모델 학습
history = model.fit(
    X_train_transformed, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# 8. 예측 예시
sample_data = pd.DataFrame([['서울특별시', '종로구', 187.49]], columns=features)
sample_data_transformed = preprocessor.transform(sample_data)
predicted_price = model.predict(sample_data_transformed)
print(f'Predicted public price: {predicted_price[0][0]:.2f}')